In [13]:
# Instalar dependencias

!pip -q install gradio
!pip -q install sentence-transformers
!pip -q install scikit-learn
!pip -q install numpy

In [14]:
# Importar librerías necesarias

import numpy as np
import gradio as gr

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [15]:
# Crear dataset JSON

datos_rag = [
    {
        "pregunta": "¿Qué cubre el producto SPP03?",
        "fichero_fuente": "2.1.3 Seguro de Protección de Pagos",
        "respuesta_modelo": "El producto SPP03 corresponde al Seguro de Protección de Pagos y cubre situaciones de incapacidad temporal del titular."
    },

    {
        "pregunta": "¿Cuál es el código del Seguro de Hogar?",
        "fichero_fuente": "2.1.2 Seguro de Hogar",
        "respuesta_modelo": "El código del Seguro de Hogar es SHG02."
    },

    {
        "pregunta": "¿Qué representa una póliza?",
        "fichero_fuente": "2.2 Componentes de una póliza",
        "respuesta_modelo": "Una póliza representa la unidad básica de gestión dentro del área de seguros."
    },

    {
        "pregunta": "¿Por qué el SME puede convertirse en un cuello de botella?",
        "fichero_fuente": "1.3 Roles principales",
        "respuesta_modelo": "Porque concentra conocimiento crítico no documentado del que dependen otros roles."
    }
]

In [17]:
# Crear documentos

documentos = []

for item in datos_rag:

    texto = f"""
Pregunta:
{item['pregunta']}

Respuesta:
{item['respuesta_modelo']}

Fuente:
{item['fichero_fuente']}
"""

    documentos.append(texto)

In [18]:
# Especificar modelo embeddings

modelo_embeddings = SentenceTransformer(
    'sentence-transformers/all-MiniLM-L6-v2'
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:
# Generar embeddings

print("Generando embeddings...")

embeddings_docs = modelo_embeddings.encode(
    documentos,
    convert_to_numpy=True
)

print("Embeddings listos.")

Generando embeddings...
Embeddings listos.


In [20]:
# Función retrieval

def responder(query):

    # Embedding query
    embedding_query = modelo_embeddings.encode(
        [query],
        convert_to_numpy=True
    )

    # Similitud
    similitudes = cosine_similarity(
        embedding_query,
        embeddings_docs
    )[0]

    # Mejor match
    idx = np.argmax(similitudes)

    mejor_documento = datos_rag[idx]

    score = similitudes[idx]

    # Control simple de alucinación
    if score < 0.45:

        return f"""
No encontrado en la base documental.

El sistema no posee suficiente información
para responder esta consulta.

Score de similitud: {round(float(score), 3)}
"""

    # Respuesta
    return f"""
Respuesta

{mejor_documento['respuesta_modelo']}

📄 Fuente:
{mejor_documento['fichero_fuente']}

📊 Score similitud:
{round(float(score), 3)}
"""

In [21]:
# Crear ejemplos

ejemplos = [

    ["¿Qué cubre el producto SPP03?"],

    ["¿Cuál es el código del Seguro de Hogar?"],

    ["¿Qué representa una póliza?"],

    ["¿Por qué el SME puede convertirse en un cuello de botella?"],

    ["¿Cuál es la capital de Francia?"],

    ["¿Qué color tiene la póliza?"]
]

In [23]:
# Crear interfaz de Gradio y lanzar

demo = gr.Interface(
    fn=responder,

    inputs=gr.Textbox(
        label="Pregunta",
        placeholder="Haz una pregunta...",
        lines=2
    ),

    outputs=gr.Textbox(
        label="Respuesta",
        lines=18,
        max_lines=30,
        show_copy_button=True
    ),

    examples=ejemplos,

    title="Demo RAG - Gestión de Conocimiento",

    description="""
Sistema RAG simplificado para consulta de
conocimiento organizacional en entidad financiera.

La respuesta se genera exclusivamente a partir
de la base documental cargada.
"""
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4ac37a0ad8f15ded2a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
